In [5]:
import pandas as pd
from pathlib import Path

# 1) ファイル存在確認
path = Path("../data/processed/Tokyo_Suginami_clean.csv")
print("CSV exists:", path.exists(), path.resolve())

df = pd.read_csv(path, encoding="utf-8-sig")
print("Loaded df shape:", df.shape)
print("Columns:", df.columns.tolist())

# 2) モデル用データ作成
use_cols = ["price", "area_sqm", "station_minutes", "building_age"]
missing_cols = [c for c in use_cols if c not in df.columns]
print("Missing required cols:", missing_cols)

df_model = df[use_cols].dropna().copy()
print("df_model shape after dropna:", df_model.shape)
print("NaN counts:\n", df[use_cols].isna().sum())

# 3) 学習・評価（必ずprintする）
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X = df_model[["area_sqm", "station_minutes", "building_age"]]
y = df_model["price"]

print("X shape:", X.shape, "y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("R2:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))

CSV exists: True C:\Users\ninav\real-estate-ai-lab\data\processed\Tokyo_Suginami_clean.csv
Loaded df shape: (2485, 6)
Columns: ['price', 'area_sqm', 'station_minutes', 'year_built', 'building_age', 'nearest_station']
Missing required cols: []
df_model shape after dropna: (2444, 4)
NaN counts:
 price               0
area_sqm            0
station_minutes    38
building_age       24
dtype: int64
X shape: (2444, 3) y shape: (2444,)
R2: 0.2658619205534931
MAE: 11842410.904630603


In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X = df_model[["area_sqm", "station_minutes", "building_age"]]
y = df_model["price"]

y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

pred_log = model.predict(X_test)

# logスケールの指標
print("R2 (log):", r2_score(y_test, pred_log))

# 元スケールに戻してMAE
pred = np.expm1(pred_log)
y_true = np.expm1(y_test)
print("MAE (yen):", mean_absolute_error(y_true, pred))

R2 (log): 0.8089106086053481
MAE (yen): 12822343.346077777


In [7]:
import numpy as np

mape = np.mean(np.abs((y_true - pred) / y_true)) * 100
print("MAPE (%):", mape)

MAPE (%): 25.335275809784143


In [8]:
df_model["price_per_sqm"] = df_model["price"] / df_model["area_sqm"]

X = df_model[["station_minutes", "building_age"]]
y = df_model["price_per_sqm"]

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X = df_model[["area_sqm", "station_minutes", "building_age"]]
y = df_model["price"]

y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

pred_log = model.predict(X_test)

# logスケールの指標
print("R2 (log):", r2_score(y_test, pred_log))

# 元スケールに戻してMAE
pred = np.expm1(pred_log)
y_true = np.expm1(y_test)
print("MAE (yen):", mean_absolute_error(y_true, pred))

R2 (log): 0.8089106086053481
MAE (yen): 12822343.346077777


In [10]:
import numpy as np

mape = np.mean(np.abs((y_true - pred) / y_true)) * 100
print("MAPE (%):", mape)

MAPE (%): 25.335275809784143


In [11]:
print("===== Model Evaluation Summary =====")
print("Linear Regression (raw price)")
print("R2:", 0.2658619205534931)
print("MAE (yen): ~11.8M")
print()
print("Linear Regression (log price)")
print("R2 (log): 0.8089106086053481")
print("MAE (yen): ~12.8M")
print("MAPE (%): 25.3")

===== Model Evaluation Summary =====
Linear Regression (raw price)
R2: 0.2658619205534931
MAE (yen): ~11.8M

Linear Regression (log price)
R2 (log): 0.8089106086053481
MAE (yen): ~12.8M
MAPE (%): 25.3


Model Evaluation and Discussion
1. Raw Price Model (Linear Regression)

本モデルでは、面積（area_sqm）、駅距離（station_minutes）、築年数（building_age）の3変数を用いて価格（price）を予測した。

R²は約0.27であり、価格変動の約27%しか説明できていない。
この結果は、杉並区内の価格差をこの3変数のみでは十分に説明できないことを示している。

特に立地差（駅ごとの価格帯の違い）を考慮していないため、モデルの説明力は限定的である。

2. Log-Transformed Price Model

価格を対数変換して回帰を行ったところ、log空間でのR²は約0.81まで改善した。

これは、価格分布の歪み（高額物件の影響）を補正できたことを意味する。
不動産価格は右に裾の長い分布を持つため、対数変換は合理的な手法である。

しかし、MAPEは約25%であり、実務水準としてはまだ誤差が大きい。
総額ベースでは依然として大きなズレが発生している。

3. Limitations of the Current Model

本モデルの主な制約は以下の通りである。

立地（駅名）を考慮していない

周辺環境や用途地域などの情報を含んでいない

非線形関係を捉えていない

そのため、「杉並区全体を均一な市場として扱う」仮定になっている。

4. Conclusion

3変数のみの単純な線形回帰でも、価格の大まかな構造は捉えられることが確認できた。
特に対数変換を用いることで、分布歪みへの対応が可能である。

一方で、実務で使用するには立地情報の組み込みや、より高度なモデルへの拡張が必要である。